In [3]:
# AUTOMATIZOVANÉ STAŽENÍ A VYKRESLENÍ PARCEL (RÚIAN / WFS INSPIRE)

import re
import requests
import pandas as pd
from lxml import etree
from pyproj import Transformer
import folium
from shapely.geometry import Polygon
from branca.element import MacroElement
from jinja2 import Template
from IPython.display import display
from folium.map import Layer
from jinja2 import Template


# =============================================================================
# 1. TŘÍDY A FUNKCE PRO VYKRESLOVÁNÍ MAPY (FOLIUM)
# =============================================================================

class BindClickRemove(MacroElement):
    """
    MacroElement, který navěsí click handler na GeoJson vrstvu.
    Po kliknutí na polygon odstraní samotný polygon i k němu příslušející
    textový popisek (marker) z jejich příslušných FeatureGroup vrstev.
    """
    def __init__(self, fg_poly_name: str, fg_text_name: str, gj_name: str, mk_name: str):
        super().__init__()
        self.fg_poly_name = fg_poly_name
        self.fg_text_name = fg_text_name
        self.gj_name = gj_name
        self.mk_name = mk_name

        self._template = Template(
            """
            {% macro script(this, kwargs) %}
            // Navěšení události click -> odstranění polygonu i popisku
            {{ this.gj_name }}.on('click', function(e) {
                {{ this.fg_poly_name }}.removeLayer({{ this.gj_name }});
                {{ this.fg_text_name }}.removeLayer({{ this.mk_name }});
            });
            {% endmacro %}
            """
        )

# Paleta výrazných barev pro vizuální odlišení parcel podle čísla LV
DISTINCT_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
]

def plot_parcels_on_map(df_parcel_data: pd.DataFrame) -> folium.Map:
    """
    Vykreslí parcely z DataFrame do interaktivní mapy Folium.
    Obsahuje oddělené vrstvy pro polygony a text (umožňuje vypínání textu).
    """
    transformer = Transformer.from_crs("EPSG:5514", "EPSG:4326", always_xy=True)

    # Ochrana: pokud chybí sloupec LV, doplníme zástupnou hodnotu
    if "LV" not in df_parcel_data.columns:
        df_parcel_data["LV"] = "Neznámé"

    # Zmapování unikátních LV na konkrétní barvy z palety
    unique_lvs = df_parcel_data["LV"].astype(str).unique()
    lv_color_map = {lv: DISTINCT_COLORS[i % len(DISTINCT_COLORS)] for i, lv in enumerate(unique_lvs)}

    items: list[tuple[Polygon, tuple[float, float], str, str]] = []
    legend_list_items = "" # Sem se bude skládat HTML pro plovoucí okno legendy

    # Zpracování geometrie a atributů pro každý řádek (parcelu)
    for idx, row in df_parcel_data.iterrows():
        posList_str = row.get("geometry_posList")

        if not posList_str or pd.isna(posList_str):
            continue

        # Převod textového seznamu souřadnic S-JTSK na GPS (WGS84)
        coords = list(map(float, posList_str.split()))
        xy_pairs = list(zip(coords[0::2], coords[1::2]))
        lon_lat_pairs = [transformer.transform(x, y) for x, y in xy_pairs]

        poly = Polygon(lon_lat_pairs)
        if not poly.is_valid or poly.is_empty:
            continue

        centroid = (poly.centroid.y, poly.centroid.x)

        # Načtení informací pro popisky
        parc_label = row.get("label", "")
        area = row.get("areaValue_m2", None)
        lv = str(row.get("LV", "Neznámé"))
        okres = row.get("okres_nazev", "Neznámý")
        ku = row.get("ku_nazev", "Neznámé")
        info_text = str(row.get("info", "")) # PŘIDÁNO: Extrakce textu info

        # Příprava URL odkazu (očištění ID od prefixu "CP.")
        gml_id_raw = str(row.get("gml_id", ""))
        clean_id = gml_id_raw.replace("CP.", "")
        url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={clean_id}"

        area_str = f"{float(area):,.0f}".replace(",", " ") if pd.notna(area) else "neznámá výměra"
        color = lv_color_map[lv]

        # Třířádkový HTML popisek nad polygonem mapy
        label_html = f"LV č. {lv}<br>{parc_label}<br>{area_str} m²"
        
        items.append((poly, centroid, label_html, color))

# PŘIDÁNO: Úprava položky do plovoucího seznamu (legendy) o nový div s poznámkou
        legend_list_items += (
            f"<li style='margin-bottom: 12px; border-bottom: 1px solid #e0e0e0; padding-bottom: 6px;'>"
            f"<span style='display:inline-block; width:14px; height:14px; background-color:{color}; "
            f"border:1px solid #333; margin-right:8px; vertical-align:middle;'></span>"
            f"<span style='vertical-align:middle; font-family:sans-serif;'>"
            f"LV č.{lv}, parc.č. <a href='{url_kn}' target='_blank' style='font-weight:bold; color:#0066cc; text-decoration:none;'>{parc_label}</a>, "
            f"{area_str} m², okres {okres}, k.ú. {ku}"
            f"</span>"
            # Přidání bloku pro doplňkové informace - odsazené, menším fontem a šedou barvou, aby nenarušovaly základní data
            f"<div style='margin-left: 26px; margin-top: 4px; font-size: 11.5px; color: #444; font-style: italic; line-height: 1.4;'>"
            f"{info_text}"
            f"</div>"
            f"</li>"
        )

    # Inicializace prázdné mapy v případě chyby
    if not items:
        print("Nebyla nalezena žádná validní geometrie, vracím defaultní mapu ČR.")
        return folium.Map(location=[49.8, 15.5], zoom_start=7)

    # Výpočet středu mapy podle načtených parcel
    avg_lat = sum(c[0] for _, c, _, _ in items) / len(items)
    avg_lon = sum(c[1] for _, c, _, _ in items) / len(items)

    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=18, tiles=None, width="100%", height="100%")

    # --- DEFINICE PODKLADOVÝCH VRSTEV ---
    # Minimalistická mapa (ideální pro zvýraznění barevných polygonů, nevyžaduje restrikční hlavičky)
    folium.TileLayer(
        tiles="CartoDB positron",
        name="Základní mapa (světlá)",
        control=True
    ).add_to(m)

    folium.raster_layers.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="© Google",
        name="Google Maps",
        overlay=False,
        control=True
    ).add_to(m)

    # Oficiální ortofoto ČÚZK
    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ORTOFOTO_WM/MapServer/tile/{z}/{y}/{x}",
        name="ČÚZK Ortofoto",
        attr="© ČÚZK",
        overlay=False,
        control=True,
        max_zoom=20,
        min_zoom=6,
        show=False,
    ).add_to(m)

    # Oficiální základní topografická mapa (ZTM ČÚZK) - spolehlivější než mapy.cz / osm
    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ZTM_WM/MapServer/tile/{z}/{y}/{x}",
        attr="© ČÚZK",
        name="Základní topografická mapa (ČÚZK)",
        overlay=False,
        control=True,
        show=False,
    ).add_to(m)


    class DynamicArcGISTileLayer(Layer):
        """
        Vlastní třída dědící z folium.map.Layer (zajišťuje správné pořadí renderování v DOMu).
        Využívá anonymní in-line třídu L.TileLayer pro bezpečné vygenerování mapy bez kolize proměnných.
        """
        _template = Template(u"""
            {% macro script(this, kwargs) %}
                var {{ this.get_name() }} = new (L.TileLayer.extend({
                    getTileUrl: function(coords) {
                        var tileSize = 256;
                        // Přepočty pro Web Mercator (EPSG:3857)
                        var initialResolution = 2 * Math.PI * 6378137 / tileSize;
                        var originShift = 2 * Math.PI * 6378137 / 2.0;
                        var resolution = initialResolution / Math.pow(2, coords.z);
                        
                        var minx = coords.x * tileSize * resolution - originShift;
                        var maxx = (coords.x + 1) * tileSize * resolution - originShift;
                        var miny = originShift - (coords.y + 1) * tileSize * resolution;
                        var maxy = originShift - coords.y * tileSize * resolution;
                        
                        var bbox = [minx, miny, maxx, maxy].join(",");
                        
                        // Z url byl odstraněn parametr '&layers=show:0', vyžádáme si kompletní sadu vrstev
                        return "{{ this.url }}?bbox=" + bbox +
                            "&bboxSR=102100&imageSR=102100&size=256,256" +
                            "&format=png32&transparent=true&f=image";
                    }
                }))({
                    opacity: {{ this.opacity }},
                    minZoom: {{ this.min_zoom }},
                    maxZoom: {{ this.max_zoom }}
                });
                
                // Bezpečné přidání do parent FeatureGroup
                {{ this.get_name() }}.addTo({{ this._parent.get_name() }});
            {% endmacro %}
        """)

        def __init__(self, url, opacity=0.5, min_zoom=10, max_zoom=18):
            super().__init__()
            self._name = 'DynamicArcGISTileLayer'
            self.url = url
            self.opacity = opacity
            self.min_zoom = min_zoom
            self.max_zoom = max_zoom

    # Přidání vrstvy do mapy (IPR Praha - export endpoint)
    arcgis_url = "https://gs-pub.praha.eu/arcgis/rest/services/pup/uzemni_plan_platny/MapServer/export"

    dynamic_fg = folium.FeatureGroup(
        name="Územní plán Prahy – plán využití",
        overlay=True,
        control=True,
        show=False,
    )
    dynamic_fg.add_child(DynamicArcGISTileLayer(arcgis_url, opacity=0.5, min_zoom=0, max_zoom=30))
    m.add_child(dynamic_fg)



    # --- DEFINICE ZOBRAZOVANÝCH DAT (POLYGONY A TEXTY) ---
    # Rozdělení do dvou nezávislých vrstev kvůli možnosti vypínat text při oddálení
    fg_poly = folium.FeatureGroup(name="Polygony parcel", show=True)
    fg_text = folium.FeatureGroup(name="Popisky parcel (text)", show=True)

    fg_poly_name = fg_poly.get_name()
    fg_text_name = fg_text.get_name()

    for poly, centroid, label_html, color in items:
        # Vykreslení polygonu s uzávěrem lambda funkce pro zachování správné barvy (c=color)
        gj = folium.GeoJson(
            data=poly.__geo_interface__,
            style_function=lambda feature, c=color: {
                "fillColor": c,
                "color": c,
                "weight": 2,
                "fillOpacity": 0.4,
            },
        ).add_to(fg_poly)

        # Vykreslení textového markeru
        mk = folium.Marker(
            location=centroid,
            draggable=True,
            icon=folium.DivIcon(
                icon_size=(150, 54),
                icon_anchor=(75, 27),
                html=(
                    '<div style="font-size:10px; font-weight:bold; line-height: 1.2;'
                    'text-align:center; color: black; '
                    'text-shadow: 2px 2px 4px white, -1px -1px 0 white, 1px -1px 0 white, -1px 1px 0 white, 1px 1px 0 white;">'
                    f"{label_html}"
                    "</div>"
                ),
            ),
        ).add_to(fg_text)

        # Propojení click eventu - smaže prvek z obou vrstev
        gj.add_child(BindClickRemove(
            fg_poly_name=fg_poly_name, 
            fg_text_name=fg_text_name, 
            gj_name=gj.get_name(), 
            mk_name=mk.get_name()
        ))

    fg_poly.add_to(m)
    fg_text.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    # --- INJEKCE PLOVOUCÍ LEGENDY ---
    legend_html_container = f"""
    <div style="position: fixed; 
                bottom: 30px; left: 30px; width: auto; max-width: 1250px; max-height: 1000px; 
                background-color: rgba(255, 255, 255, 0.95); border: 2px solid #aaa; z-index: 9999; 
                overflow-y: auto; padding: 10px; border-radius: 8px; box-shadow: 3px 3px 10px rgba(0,0,0,0.3);">
        <h4 style="margin-top: 0; margin-bottom: 10px; font-family: sans-serif; border-bottom: 2px solid #444; padding-bottom: 5px;">
            Zobrazené pozemky
        </h4>
        <ul style="list-style-type: none; padding-left: 0; margin: 0; font-size: 10px;">
            {legend_list_items}
        </ul>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html_container))

    return m

# =============================================================================
# 2. FUNKCE PRO DOTAZOVÁNÍ RÚIAN A INSPIRE (ZÁKLADNÍ BEZE ZMĚN)
# =============================================================================

def convert_to_gps(x: float, y: float, source_epsg: str = "EPSG:5514") -> tuple[float, float]:
    transformer = Transformer.from_crs(source_epsg, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)
    return lon, lat

def get_parcel_data(okres_nazev: str, kat_uzemi_nazev: str, parcel_number: str) -> pd.DataFrame:
    PRAGUE_OBEC_KOD = 554782
    PRAGUE_VUSC_KOD = 19
    PRAGUE_ALIASES = {"praha", "hlavni mesto praha", "hlavní město praha", "praha-mesto", "praha město"}

    def _norm(s: str) -> str: return (s or "").strip().lower()
    def _is_prague_okres(name: str) -> bool: return _norm(name) in PRAGUE_ALIASES
    def _sql_escape(s: str) -> str: return (s or "").replace("'", "''")

    base_url_candidates = [
        "https://ags.cuzk.gov.cz/arcgis/rest/services/RUIAN/Prohlizeci_sluzba_nad_daty_RUIAN/MapServer",
        "https://ags.cuzk.cz/ArcGIS/rest/services/RUIAN/MapServer",
    ]

    session = requests.Session()

    def arcgis_query(base_url: str, layer_id: int, where: str, out_fields: str = "*") -> dict:
        params = {"where": where, "outFields": out_fields, "returnGeometry": "false", "f": "json"}
        r = session.get(f"{base_url}/{layer_id}/query", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and data.get("error"):
            msg = data["error"].get("message", "ArcGIS error")
            raise RuntimeError(f"ArcGIS query error: {msg}")
        return data

    def arcgis_query_first_ok(layer_id: int, where: str, out_fields: str = "*") -> dict:
        last_err = None
        for base_url in base_url_candidates:
            try:
                return arcgis_query(base_url, layer_id, where, out_fields=out_fields)
            except Exception as e:
                last_err = e
        raise RuntimeError(f"Nepodařilo se dotázat RÚIAN ArcGIS. Poslední chyba: {last_err}")

    praha_mode = _is_prague_okres(okres_nazev)
    okres_kod, okres_attrs = None, None

    if not praha_mode:
        okres_data = arcgis_query_first_ok(layer_id=15, where=f"nazev = '{_sql_escape(okres_nazev)}'", out_fields="*")
        if not okres_data.get("features"):
            raise ValueError(f"Okres '{okres_nazev}' nebyl nalezen.")
        okres_attrs = okres_data["features"][0]["attributes"]
        okres_kod = okres_attrs.get("kod")

    ku_data = arcgis_query_first_ok(layer_id=7, where=f"nazev LIKE '{_sql_escape(kat_uzemi_nazev)}%'", out_fields="kod,nazev,obec")
    if not ku_data.get("features"):
        raise ValueError(f"Katastrální území podobné '{kat_uzemi_nazev}' nebylo nalezeno.")

    valid_ku = []
    for f in ku_data["features"]:
        ku_atr = f["attributes"]
        obec_kod = ku_atr.get("obec")
        if obec_kod is None: continue

        obec_data = arcgis_query_first_ok(layer_id=12, where=f"kod = {int(obec_kod)}", out_fields="kod,nazev,okres")
        if not obec_data.get("features"): continue
        obec_atr = obec_data["features"][0]["attributes"]

        if praha_mode:
            if int(obec_atr.get("kod", -1)) == PRAGUE_OBEC_KOD:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})
        else:
            if obec_atr.get("okres") == okres_kod:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})

    if not valid_ku:
        raise ValueError(f"Nebylo nalezeno žádné katastrální území.")

    selected_ku = valid_ku[0]

    if praha_mode:
        vusc_kod = PRAGUE_VUSC_KOD
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = None
        okres_nazev_out = okres_nazev
    else:
        okres2 = arcgis_query_first_ok(layer_id=15, where=f"kod = {int(okres_kod)}", out_fields="kod,nazev,vusc")
        okres_attrs = okres2["features"][0]["attributes"]
        vusc_kod = okres_attrs.get("vusc")
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = okres_attrs.get("kod")
        okres_nazev_out = okres_attrs.get("nazev")

    # WFS INSPIRE GetParcel
    params_wfs = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "storedQuery_id": "GetParcel", "UPPER_ZONING_ID": selected_ku["ku_kod"], "TEXT": parcel_number
    }
    resp_wfs = session.get("https://services.cuzk.cz/wfs/inspire-CP-wfs.asp", params=params_wfs, timeout=30)
    resp_wfs.raise_for_status()

    tree = etree.fromstring(resp_wfs.content)
    ns = {"wfs": "http://www.opengis.net/wfs/2.0", "gml": "http://www.opengis.net/gml/3.2", "CP": "http://inspire.ec.europa.eu/schemas/cp/4.0", "base": "http://inspire.ec.europa.eu/schemas/base/3.3"}

    parcel_elem = tree.find(".//CP:CadastralParcel", namespaces=ns)
    if parcel_elem is None:
        raise ValueError(f"Parcela {parcel_number} nebyla nalezena.")

    def get_text(elem, path):
        sub = elem.find(path, namespaces=ns)
        return sub.text.strip() if sub is not None and sub.text else None

    parcel_data = {
        "gml_id": parcel_elem.get("{http://www.opengis.net/gml/3.2}id"),
        "areaValue_m2": float(get_text(parcel_elem, "CP:areaValue") or 0),
        "beginLifespanVersion": get_text(parcel_elem, "CP:beginLifespanVersion"),
        "endLifespanVersion": get_text(parcel_elem, "CP:endLifespanVersion"),
        "label": get_text(parcel_elem, "CP:label"),
        "nationalCadastralReference": get_text(parcel_elem, "CP:nationalCadastralReference"),
        "inspire_localId": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:localId"),
        "inspire_namespace": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:namespace"),
        "refPoint_x": None, "refPoint_y": None, "refPoint_lon": None, "refPoint_lat": None,
        "geometry_posList": get_text(parcel_elem, "CP:geometry/gml:Polygon/gml:exterior/gml:LinearRing/gml:posList"),
        "ku_kod": selected_ku["ku_kod"], "ku_nazev": selected_ku["ku_nazev"],
        "obec_kod": selected_ku["obec_kod"], "obec_nazev": selected_ku["obec_nazev"],
        "okres_kod": okres_kod_out, "okres_nazev": okres_nazev_out,
        "vusc_kod": vusc_attrs.get("kod"), "vusc_nazev": vusc_attrs.get("nazev"),
    }

    ref_point = get_text(parcel_elem, "CP:referencePoint/gml:Point/gml:pos")
    if ref_point:
        coords = ref_point.split()
        if len(coords) >= 2:
            parcel_data["refPoint_x"] = float(coords[0])
            parcel_data["refPoint_y"] = float(coords[1])
            lon, lat = convert_to_gps(float(coords[0]), float(coords[1]))
            parcel_data["refPoint_lon"], parcel_data["refPoint_lat"] = lon, lat

    return pd.DataFrame([parcel_data])

# =============================================================================
# 3. HLAVNÍ BLOK: ZPRACOVÁNÍ VSTUPŮ A VÝSTUP
# =============================================================================

# Vstupní data (okres, katastrální území, parcelní číslo, číslo LV)
parcely = [


    ("Praha", "Hostivař", "1714/4", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/5", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/6", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),    
    ("Praha", "Hostivař", "1714/7", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/8", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),

    ("Praha", "Hostivař", "1714/3", "804","Oceňovaný pozemek"),

    # Přidejte další parcely podle potřeby...
]

dfs = []

print("Načítám data z API ČÚZK...")
# Iterace skrz vstupní tuple o PĚTI prvcích
for okres, ku, parc, lv, info in parcely:
    df_one = get_parcel_data(okres, ku, parc)
    
    # Přidání LV a info jako nových sloupců pro další zpracování
    df_one["LV"] = str(lv) 
    df_one["info"] = str(info) # PŘIDÁNO: Načtení informací do DataFramu
    
    dfs.append(df_one)

# Sloučení všech nalezených parcel do jednoho DataFramu
df_parcel_data = pd.concat(dfs, ignore_index=True)

# Příprava dat pro Excel
df_export = (
    df_parcel_data
    .assign(
        parcelni_cislo=lambda d: d["label"],
        lat=lambda d: d["refPoint_lat"],
        lon=lambda d: d["refPoint_lon"],
    )
    # PŘIDÁNO: Zahrnutí sloupce 'info' do konečného exportu
    .loc[:, ["okres_nazev", "ku_nazev", "obec_nazev", "parcelni_cislo", "LV", "info", "lat", "lon"]]
    .rename(columns={
        "okres_nazev": "okres",
        "ku_nazev": "katastralni_uzemi",
        "obec_nazev": "obec",
    })
)

df_export["lat"] = df_export["lat"].round(8)
df_export["lon"] = df_export["lon"].round(8)

# Uložení DataFramu do Excelu
out_path = "parcely_gps.xlsx"
df_export.to_excel(out_path, index=False, sheet_name="parcely_gps")
print(f"Data uložena do: {out_path}")

# Vykreslení interaktivní mapy
print("Generuji mapu...")
m = plot_parcels_on_map(df_parcel_data)

# Uložení mapy do souboru
html_file = "mapa_parcel.html"
m.save(html_file)
print(f"Interaktivní mapa uložena do: {html_file}")

# Zobrazení mapy přímo ve VS Code (Jupyter)
#display(m)
#m.show_in_browser()

from IPython.display import HTML, display

# Vygeneruje čisté HTML z objektu mapy a vynutí jeho zobrazení pod buňkou
display(HTML(m._repr_html_()))

Načítám data z API ČÚZK...
Data uložena do: parcely_gps.xlsx
Generuji mapu...
Interaktivní mapa uložena do: mapa_parcel.html


In [4]:
#  VERZE 2 s tahanim dat o cenach z mych DB


# AUTOMATIZOVANÉ STAŽENÍ A VYKRESLENÍ PARCEL (RÚIAN / WFS INSPIRE)

import re
import requests
import pandas as pd
from lxml import etree
from pyproj import Transformer
import folium
from shapely.geometry import Polygon
from branca.element import MacroElement
from jinja2 import Template
from folium.map import Layer
from IPython.display import HTML, display
import urllib.parse
from sqlalchemy import create_engine

# =============================================================================
# 0. PŘIPOJENÍ K DATABÁZI A POMOCNÉ FUNKCE PRO HISTORII
# =============================================================================

# Připojení k DB Valuo pomocí SQLAlchemy
params_conn = urllib.parse.quote_plus(
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost;"
    "Database=VALUO;"
    "Trusted_Connection=yes;"
)
connection_url = f"mssql+pyodbc:///?odbc_connect={params_conn}"
engine = create_engine(connection_url)

def get_valuo_history(okres: str, ku: str, parcelni_cislo: str, db_engine) -> str:
    """
    Vyhledá historii parcely v DB Valuo.
    Vytváří HTML odkazy do katastru pomocí sloupce 'gml_id' z tabulky KN_parcel_data.
    """
    db_okres = okres
    if okres.lower().strip() in ["praha", "hlavni mesto praha", "praha-mesto", "praha město"]:
        db_okres = "Hlavní město Praha"
        
    # --- POMOCNÁ FUNKCE (ROZBALOVACÍ SEZNAM S ODKAZY NA KN) ---
    def zformatuj_parcely(kombi_series, max_zobrazeno=5):
        # Rozdělení na unikátní záznamy
        items = set([str(i).strip() for i in kombi_series.dropna() if str(i).strip() and str(i).strip() != '|'])
        if not items:
            return ""
        
        links = []
        # Třídění čistě podle parcelního čísla
        for item in sorted(items, key=lambda x: x.split('|')[0]):
            parts = item.split('|')
            p_num = parts[0]
            p_kod = parts[1] if len(parts) > 1 else ""
            
            # Pokud máme čisté číselné ID, vyrobíme odkaz do KN
            if p_kod and p_kod.isdigit():
                url = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={p_kod}"
                links.append(f"<a href='{url}' target='_blank' style='color:#0066cc; text-decoration:none; font-weight:bold;' title='Otevřít v KN'>{p_num}</a>")
            else:
                links.append(p_num) # Fallback na prostý text, pokud by ID chybělo
                
        plny_seznam = ", ".join(links)
        
        if len(links) > max_zobrazeno:
            nahled = ", ".join(links[:max_zobrazeno])
            zbyva = len(links) - max_zobrazeno
            return (
                f"<details style='cursor: pointer; margin-top: 2px;'>"
                f"<summary style='outline: none; color: #555;'>{nahled} ... <b style='color: #d9534f;'>(+ {zbyva} rozbalit)</b></summary>"
                f"<div style='margin-top: 4px; padding: 6px; border-left: 2px solid #d9534f; background: #f4f4f4; color: #333; line-height: 1.4; word-wrap: break-word;'>"
                f"{plny_seznam}</div>"
                f"</details>"
            )
        return plny_seznam
    # -------------------------------------------------------------
    
    # 1. Krok: Získání čísel vkladů
    query_vklady = f"""
        SELECT DISTINCT v.cislo_vkladu
        FROM Valuo_data v
        JOIN KN_parcel_data p ON v.id = p.id_valuo
        WHERE v.okres = '{db_okres}' 
          AND v.kat_uzemi = '{ku}' 
          AND p.parcel_number = '{parcelni_cislo}'
    """
    try:
        df_vklady = pd.read_sql(query_vklady, db_engine)
    except Exception as e:
        return f"<div style='color:red;'>Chyba prvního dotazu: {e}</div>"
        
    if df_vklady.empty:
        return "<i style='color:gray;'>Záznam v databázi Valuo nenalezen.</i>"
        
    vklady_list = tuple(df_vklady['cislo_vkladu'].tolist())
    vklady_str = f"('{vklady_list[0]}')" if len(vklady_list) == 1 else str(vklady_list)
        
    # 2. Krok: UPRAVENÝ DOTAZ PRO GML_ID (Oříznutí prefixu 'CP.' přímo v SQL)
    query_details = f"""
        SELECT 
            v.id, 
            v.cislo_vkladu, 
            CONVERT(VARCHAR(10), v.datum_podani, 104) AS datum_podani, 
            CAST(v.cenovy_udaj AS FLOAT) AS cenovy_udaj, 
            v.nemovitost, 
            CAST(v.plocha AS FLOAT) AS plocha,
            CAST(p.parcel_number AS VARCHAR(100)) + '|' + ISNULL(REPLACE(CAST(p.gml_id AS VARCHAR(100)), 'CP.', ''), '') AS parcel_data_kombi
        FROM Valuo_data v
        LEFT JOIN KN_parcel_data p ON v.id = p.id_valuo
        WHERE v.cislo_vkladu IN {vklady_str}
    """
    
    try:
        df_details = pd.read_sql(query_details, db_engine)
    except Exception as e:
        return f"<div style='color:red;'>Chyba pro stahování detailů vkladu: {e}</div>"
    
    # 3. Krok: Zpracování a výstup
    html_output = ""
    for vklad, group in df_details.groupby('cislo_vkladu'):
        datum = str(group['datum_podani'].iloc[0]) if pd.notnull(group['datum_podani'].iloc[0]) else "Neznámé"
        cena = float(group['cenovy_udaj'].max()) 
        
        stats = group.groupby('nemovitost').agg(
            pocet=('id', 'count'),
            plocha_sum=('plocha', 'sum'),
            seznam_parcel=('parcel_data_kombi', lambda x: zformatuj_parcely(x, max_zobrazeno=5))
        ).reset_index()
        
        celkova_plocha = stats['plocha_sum'].sum()
        jc = cena / celkova_plocha if celkova_plocha > 0 else 0
        
        html_output += f"<div style='background: #f9f9f9; border: 1px solid #ccc; padding: 5px; margin-bottom: 5px;'>"

        # Odkaz směřuje na formulář vyhledání, jelikož přímé URL bez interního ID ČÚZK neumožňuje
        url_rizeni = "https://nahlizenidokn.cuzk.gov.cz/VyberRizeni.aspx"
        odkaz_vklad = f"<a href='{url_rizeni}' target='_blank' style='color:#0066cc; text-decoration:none; font-weight:bold;' title='Zkopírujte číslo a otevřete vyhledávání v KN'>{vklad}</a>"
        html_output += f"<b>Řízení:</b> {odkaz_vklad} (ze dne {datum})<br>"

        html_output += f"<b>Kupní cena:</b> {cena:,.0f} Kč<br>".replace(',', ' ')
        html_output += f"<b>JC: {jc:,.0f} Kč/m²</b> (z plochy {celkova_plocha:,.0f} m²)<br>".replace(',', ' ')
        html_output += "<i style='font-size: 11px;'>Složení transakce:</i><br>"
        
        for _, r in stats.iterrows():
            html_output += f"<span style='font-size: 11px;'>- {r['nemovitost']}: {r['pocet']}x ({r['plocha_sum']:,.0f} m²)</span><br>".replace(',', ' ')
            if r['seznam_parcel']:
                html_output += (
                    f"<div style='font-size: 10px; margin-left: 12px; margin-top: 1px; "
                    f"margin-bottom: 4px;'>Parc. č.: {r['seznam_parcel']}</div>"
                )
        html_output += "</div>"
        
    return html_output
# =============================================================================
# 1. TŘÍDY A FUNKCE PRO VYKRESLOVÁNÍ MAPY (FOLIUM)
# =============================================================================

class BindClickRemove(MacroElement):
    """
    MacroElement pro odstranění polygonu.
    UPRAVENO: Nyní se maže PRAVÝM kliknutím ('contextmenu'), aby levé kliknutí otevíralo okno.
    """
    def __init__(self, fg_poly_name: str, fg_text_name: str, gj_name: str, mk_name: str):
        super().__init__()
        self.fg_poly_name = fg_poly_name
        self.fg_text_name = fg_text_name
        self.gj_name = gj_name
        self.mk_name = mk_name

        self._template = Template(
            """
            {% macro script(this, kwargs) %}
            // ZMĚNA: 'contextmenu' znamená pravé tlačítko myši
            {{ this.gj_name }}.on('contextmenu', function(e) {
                {{ this.fg_poly_name }}.removeLayer({{ this.gj_name }});
                {{ this.fg_text_name }}.removeLayer({{ this.mk_name }});
            });
            {% endmacro %}
            """
        )

DISTINCT_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
]

# =============================================================================
# 1. TŘÍDY A FUNKCE PRO VYKRESLOVÁNÍ MAPY (FOLIUM)
# =============================================================================

class BindClickRemove(MacroElement):
    """
    MacroElement, který navěsí click handler na GeoJson vrstvu.
    UPRAVENO: Odstranění se nyní spouští PRAVÝM tlačítkem myši ('contextmenu').
    """
    def __init__(self, fg_poly_name: str, fg_text_name: str, gj_name: str, mk_name: str):
        super().__init__()
        self.fg_poly_name = fg_poly_name
        self.fg_text_name = fg_text_name
        self.gj_name = gj_name
        self.mk_name = mk_name

        self._template = Template(
            """
            {% macro script(this, kwargs) %}
            // Navěšení události contextmenu (pravý klik) -> odstranění polygonu i popisku
            {{ this.gj_name }}.on('contextmenu', function(e) {
                {{ this.fg_poly_name }}.removeLayer({{ this.gj_name }});
                {{ this.fg_text_name }}.removeLayer({{ this.mk_name }});
            });
            {% endmacro %}
            """
        )

# Paleta výrazných barev pro vizuální odlišení parcel podle čísla LV
DISTINCT_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe",
    "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000",
    "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080"
]

def plot_parcels_on_map(df_parcel_data: pd.DataFrame) -> folium.Map:
    """
    Vykreslí parcely z DataFrame do interaktivní mapy Folium.
    Obsahuje oddělené vrstvy pro polygony a text, historii z DB a plovoucí legendu.
    """
    transformer = Transformer.from_crs("EPSG:5514", "EPSG:4326", always_xy=True)

    if "LV" not in df_parcel_data.columns:
        df_parcel_data["LV"] = "Neznámé"

    unique_lvs = df_parcel_data["LV"].astype(str).unique()
    lv_color_map = {lv: DISTINCT_COLORS[i % len(DISTINCT_COLORS)] for i, lv in enumerate(unique_lvs)}

    items: list[tuple[Polygon, tuple[float, float], str, str, str]] = []
    legend_list_items = "" 

    for idx, row in df_parcel_data.iterrows():
        posList_str = row.get("geometry_posList")

        if not posList_str or pd.isna(posList_str):
            continue

        coords = list(map(float, posList_str.split()))
        xy_pairs = list(zip(coords[0::2], coords[1::2]))
        lon_lat_pairs = [transformer.transform(x, y) for x, y in xy_pairs]

        poly = Polygon(lon_lat_pairs)
        if not poly.is_valid or poly.is_empty:
            continue

        centroid = (poly.centroid.y, poly.centroid.x)

        parc_label = row.get("label", "")
        area = row.get("areaValue_m2", None)
        lv = str(row.get("LV", "Neznámé"))
        okres = row.get("okres_nazev", "Neznámý")
        ku = row.get("ku_nazev", "Neznámé")
        ku_kod = row.get("ku_kod", "")
        obec_nazev = row.get("obec_nazev", "Neznámá")
        obec_kod = row.get("obec_kod", "")
        info_text = str(row.get("info", "")) 
        druh_pozemku = row.get("druh_pozemku", "Nezjištěno")

        color = lv_color_map[lv]
        area_str = f"{float(area):,.0f}".replace(",", " ") if pd.notna(area) else "neznámá výměra"

        gml_id_raw = str(row.get("gml_id", ""))
        clean_id = gml_id_raw.replace("CP.", "")
        url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={clean_id}"

        db_info_html = get_valuo_history(okres, ku, parc_label, engine)
        
# Kompletní HTML pro rozklikávací okno (Popup)
        popup_html = f"""
        <div style='font-family: sans-serif; font-size: 13px; width: 340px; line-height: 1.4;'>
            <h4 style='margin: 0 0 5px 0; border-bottom: 2px solid {color}; padding-bottom: 3px;'>
                Parcela č. <a href='{url_kn}' target='_blank' style='color:#0066cc; text-decoration:none;' title='Otevřít v Nahlížení do KN'>{parc_label}</a> (LV: {lv})
            </h4>
            <b>Druh pozemku:</b> {druh_pozemku}<br>
            <b>Výměra:</b> {area_str} m²<br>
            <b>K.Ú.:</b> {ku} ({ku_kod})<br>
            <b>Obec:</b> {obec_nazev} ({obec_kod})<br>
            <b>Okres:</b> {okres}<br>
            <hr style='border: 0; border-top: 1px solid #ccc; margin: 8px 0;'>
            <h5 style='margin: 0 0 5px 0;'>Historie transakcí (Valuo DB)</h5>
            {db_info_html}
        </div>
        """

        label_html = f"LV č. {lv}<br>{parc_label}<br>{area_str} m²"
        
        items.append((poly, centroid, label_html, color, popup_html))

        legend_list_items += (
            f"<li style='margin-bottom: 12px; border-bottom: 1px solid #e0e0e0; padding-bottom: 6px;'>"
            f"<span style='display:inline-block; width:14px; height:14px; background-color:{color}; "
            f"border:1px solid #333; margin-right:8px; vertical-align:middle;'></span>"
            f"<span style='vertical-align:middle; font-family:sans-serif;'>"
            f"LV č.{lv}, parc.č. <a href='{url_kn}' target='_blank' style='font-weight:bold; color:#0066cc; text-decoration:none;'>{parc_label}</a>, "
            f"{area_str} m², okres {okres}, k.ú. {ku}"
            f"</span>"
            f"<div style='margin-left: 26px; margin-top: 4px; font-size: 11.5px; color: #444; font-style: italic; line-height: 1.4;'>"
            f"{info_text}"
            f"</div>"
            f"</li>"
        )

    if not items:
        print("Nebyla nalezena žádná validní geometrie, vracím defaultní mapu ČR.")
        return folium.Map(location=[49.8, 15.5], zoom_start=7)

    avg_lat = sum(c[0] for _, c, _, _, _ in items) / len(items)
    avg_lon = sum(c[1] for _, c, _, _, _ in items) / len(items)

    m = folium.Map(
        location=[avg_lat, avg_lon], 
        zoom_start=18, 
        tiles=None, 
        width="100%", 
        height="100%", 
        closePopupOnClick=False 
    )

    folium.TileLayer("CartoDB positron", name="Základní mapa (světlá)", control=True).add_to(m)

    folium.raster_layers.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="© Google", name="Google Maps", overlay=False, control=True
    ).add_to(m)

    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ORTOFOTO_WM/MapServer/tile/{z}/{y}/{x}",
        name="ČÚZK Ortofoto", attr="© ČÚZK", overlay=False, control=True, max_zoom=20, min_zoom=6, show=False,
    ).add_to(m)

    folium.raster_layers.TileLayer(
        tiles="https://ags.cuzk.gov.cz/arcgis1/rest/services/ZTM_WM/MapServer/tile/{z}/{y}/{x}",
        attr="© ČÚZK", name="Základní topografická mapa (ČÚZK)", overlay=False, control=True, show=False,
    ).add_to(m)

    class DynamicArcGISTileLayer(Layer):
        _template = Template(u"""
            {% macro script(this, kwargs) %}
                var {{ this.get_name() }} = new (L.TileLayer.extend({
                    getTileUrl: function(coords) {
                        var tileSize = 256;
                        var initialResolution = 2 * Math.PI * 6378137 / tileSize;
                        var originShift = 2 * Math.PI * 6378137 / 2.0;
                        var resolution = initialResolution / Math.pow(2, coords.z);
                        var minx = coords.x * tileSize * resolution - originShift;
                        var maxx = (coords.x + 1) * tileSize * resolution - originShift;
                        var miny = originShift - (coords.y + 1) * tileSize * resolution;
                        var maxy = originShift - coords.y * tileSize * resolution;
                        var bbox = [minx, miny, maxx, maxy].join(",");
                        return "{{ this.url }}?bbox=" + bbox +
                            "&bboxSR=102100&imageSR=102100&size=256,256" +
                            "&format=png32&transparent=true&f=image";
                    }
                }))({ opacity: {{ this.opacity }}, minZoom: {{ this.min_zoom }}, maxZoom: {{ this.max_zoom }} });
                {{ this.get_name() }}.addTo({{ this._parent.get_name() }});
            {% endmacro %}
        """)

        def __init__(self, url, opacity=0.5, min_zoom=10, max_zoom=18):
            super().__init__()
            self._name = 'DynamicArcGISTileLayer'
            self.url = url; self.opacity = opacity; self.min_zoom = min_zoom; self.max_zoom = max_zoom

    arcgis_url = "https://gs-pub.praha.eu/arcgis/rest/services/pup/uzemni_plan_platny/MapServer/export"
    dynamic_fg = folium.FeatureGroup(name="Územní plán Prahy – plán využití", overlay=True, control=True, show=False)
    dynamic_fg.add_child(DynamicArcGISTileLayer(arcgis_url, opacity=0.5, min_zoom=0, max_zoom=30))
    m.add_child(dynamic_fg)

    fg_poly = folium.FeatureGroup(name="Polygony parcel", show=True)
    fg_text = folium.FeatureGroup(name="Popisky parcel (text)", show=True)

    fg_poly_name = fg_poly.get_name()
    fg_text_name = fg_text.get_name()

    for poly, centroid, label_html, color, popup_html in items:
        
        # ---> ZDE JE APLIKOVÁNA OPRAVA Č. 2 <---
        # Přidáno auto_close=False pro zachování více otevřených oken
        popup_okno = folium.Popup(html=popup_html, max_width=420, max_height=350, auto_close=False)

        gj = folium.GeoJson(
            data=poly.__geo_interface__,
            style_function=lambda feature, c=color: {
                "fillColor": c,
                "color": c,
                "weight": 2,
                "fillOpacity": 0.4,
            },
            tooltip="<b>Levý klik:</b> Detaily a historie DB <br><b>Pravý klik:</b> Smazat polygon",
            popup=popup_okno
        ).add_to(fg_poly)

        mk = folium.Marker(
            location=centroid,
            draggable=True,
            icon=folium.DivIcon(
                icon_size=(150, 54),
                icon_anchor=(75, 27),
                html=(
                    '<div style="font-size:10px; font-weight:bold; line-height: 1.2;'
                    'text-align:center; color: black; '
                    'text-shadow: 2px 2px 4px white, -1px -1px 0 white, 1px -1px 0 white, -1px 1px 0 white, 1px 1px 0 white;">'
                    f"{label_html}"
                    "</div>"
                ),
            ),
        ).add_to(fg_text)

        gj.add_child(BindClickRemove(
            fg_poly_name=fg_poly_name, 
            fg_text_name=fg_text_name, 
            gj_name=gj.get_name(), 
            mk_name=mk.get_name()
        ))

    fg_poly.add_to(m)
    fg_text.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    legend_html_container = f"""
    <div style="position: fixed; 
                bottom: 30px; left: 30px; width: auto; max-width: 1250px; max-height: 1000px; 
                background-color: rgba(255, 255, 255, 0.95); border: 2px solid #aaa; z-index: 9999; 
                overflow-y: auto; padding: 10px; border-radius: 8px; box-shadow: 3px 3px 10px rgba(0,0,0,0.3);">
        <h4 style="margin-top: 0; margin-bottom: 10px; font-family: sans-serif; border-bottom: 2px solid #444; padding-bottom: 5px;">
            Zobrazené pozemky
        </h4>
        <ul style="list-style-type: none; padding-left: 0; margin: 0; font-size: 10px;">
            {legend_list_items}
        </ul>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html_container))

    return m


# =============================================================================
# 2. FUNKCE PRO DOTAZOVÁNÍ RÚIAN A INSPIRE
# =============================================================================

def convert_to_gps(x: float, y: float, source_epsg: str = "EPSG:5514") -> tuple[float, float]:
    transformer = Transformer.from_crs(source_epsg, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)
    return lon, lat

def get_parcel_data(okres_nazev: str, kat_uzemi_nazev: str, parcel_number: str) -> pd.DataFrame:
    PRAGUE_OBEC_KOD = 554782
    PRAGUE_VUSC_KOD = 19
    PRAGUE_ALIASES = {"praha", "hlavni mesto praha", "hlavní město praha", "praha-mesto", "praha město"}

    def _norm(s: str) -> str: return (s or "").strip().lower()
    def _is_prague_okres(name: str) -> bool: return _norm(name) in PRAGUE_ALIASES
    def _sql_escape(s: str) -> str: return (s or "").replace("'", "''")

    base_url_candidates = [
        "https://ags.cuzk.gov.cz/arcgis/rest/services/RUIAN/Prohlizeci_sluzba_nad_daty_RUIAN/MapServer",
        "https://ags.cuzk.cz/ArcGIS/rest/services/RUIAN/MapServer",
    ]

    session = requests.Session()

    def arcgis_query(base_url: str, layer_id: int, where: str, out_fields: str = "*") -> dict:
        params = {"where": where, "outFields": out_fields, "returnGeometry": "false", "f": "json"}
        r = session.get(f"{base_url}/{layer_id}/query", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, dict) and data.get("error"):
            msg = data["error"].get("message", "ArcGIS error")
            raise RuntimeError(f"ArcGIS query error: {msg}")
        return data

    def arcgis_query_first_ok(layer_id: int, where: str, out_fields: str = "*") -> dict:
        last_err = None
        for base_url in base_url_candidates:
            try:
                return arcgis_query(base_url, layer_id, where, out_fields=out_fields)
            except Exception as e:
                last_err = e
        raise RuntimeError(f"Nepodařilo se dotázat RÚIAN ArcGIS. Poslední chyba: {last_err}")

    praha_mode = _is_prague_okres(okres_nazev)
    okres_kod, okres_attrs = None, None

    if not praha_mode:
        okres_data = arcgis_query_first_ok(layer_id=15, where=f"nazev = '{_sql_escape(okres_nazev)}'", out_fields="*")
        if not okres_data.get("features"): raise ValueError(f"Okres '{okres_nazev}' nebyl nalezen.")
        okres_attrs = okres_data["features"][0]["attributes"]
        okres_kod = okres_attrs.get("kod")

    ku_data = arcgis_query_first_ok(layer_id=7, where=f"nazev LIKE '{_sql_escape(kat_uzemi_nazev)}%'", out_fields="kod,nazev,obec")
    if not ku_data.get("features"): raise ValueError(f"Katastrální území podobné '{kat_uzemi_nazev}' nebylo nalezeno.")

    valid_ku = []
    for f in ku_data["features"]:
        ku_atr = f["attributes"]
        obec_kod = ku_atr.get("obec")
        if obec_kod is None: continue

        obec_data = arcgis_query_first_ok(layer_id=12, where=f"kod = {int(obec_kod)}", out_fields="kod,nazev,okres")
        if not obec_data.get("features"): continue
        obec_atr = obec_data["features"][0]["attributes"]

        if praha_mode:
            if int(obec_atr.get("kod", -1)) == PRAGUE_OBEC_KOD:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})
        else:
            if obec_atr.get("okres") == okres_kod:
                valid_ku.append({"ku_kod": ku_atr.get("kod"), "ku_nazev": ku_atr.get("nazev"), "obec_kod": obec_atr.get("kod"), "obec_nazev": obec_atr.get("nazev")})

    if not valid_ku: raise ValueError(f"Nebylo nalezeno žádné katastrální území.")
    selected_ku = valid_ku[0]

    if praha_mode:
        vusc_kod = PRAGUE_VUSC_KOD
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = None
        okres_nazev_out = okres_nazev
    else:
        okres2 = arcgis_query_first_ok(layer_id=15, where=f"kod = {int(okres_kod)}", out_fields="kod,nazev,vusc")
        okres_attrs = okres2["features"][0]["attributes"]
        vusc_kod = okres_attrs.get("vusc")
        vusc_data = arcgis_query_first_ok(layer_id=17, where=f"kod = {int(vusc_kod)}", out_fields="kod,nazev")
        vusc_attrs = vusc_data["features"][0]["attributes"]
        okres_kod_out = okres_attrs.get("kod")
        okres_nazev_out = okres_attrs.get("nazev")

    params_wfs = {
        "service": "WFS", "version": "2.0.0", "request": "GetFeature",
        "storedQuery_id": "GetParcel", "UPPER_ZONING_ID": selected_ku["ku_kod"], "TEXT": parcel_number
    }
    resp_wfs = session.get("https://services.cuzk.cz/wfs/inspire-CP-wfs.asp", params=params_wfs, timeout=30)
    resp_wfs.raise_for_status()

    tree = etree.fromstring(resp_wfs.content)
    ns = {"wfs": "http://www.opengis.net/wfs/2.0", "gml": "http://www.opengis.net/gml/3.2", "CP": "http://inspire.ec.europa.eu/schemas/cp/4.0", "base": "http://inspire.ec.europa.eu/schemas/base/3.3"}

    parcel_elem = tree.find(".//CP:CadastralParcel", namespaces=ns)
    if parcel_elem is None: raise ValueError(f"Parcela {parcel_number} nebyla nalezena.")

    def get_text(elem, path):
        sub = elem.find(path, namespaces=ns)
        return sub.text.strip() if sub is not None and sub.text else None

    parcel_data = {
        "gml_id": parcel_elem.get("{http://www.opengis.net/gml/3.2}id"),
        "areaValue_m2": float(get_text(parcel_elem, "CP:areaValue") or 0),
        "beginLifespanVersion": get_text(parcel_elem, "CP:beginLifespanVersion"),
        "endLifespanVersion": get_text(parcel_elem, "CP:endLifespanVersion"),
        "label": get_text(parcel_elem, "CP:label"),
        "nationalCadastralReference": get_text(parcel_elem, "CP:nationalCadastralReference"),
        "inspire_localId": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:localId"),
        "inspire_namespace": get_text(parcel_elem, "CP:inspireId/base:Identifier/base:namespace"),
        "refPoint_x": None, "refPoint_y": None, "refPoint_lon": None, "refPoint_lat": None,
        "geometry_posList": get_text(parcel_elem, "CP:geometry/gml:Polygon/gml:exterior/gml:LinearRing/gml:posList"),
        "ku_kod": selected_ku["ku_kod"], "ku_nazev": selected_ku["ku_nazev"],
        "obec_kod": selected_ku["obec_kod"], "obec_nazev": selected_ku["obec_nazev"],
        "okres_kod": okres_kod_out, "okres_nazev": okres_nazev_out,
        "vusc_kod": vusc_attrs.get("kod"), "vusc_nazev": vusc_attrs.get("nazev"),
    }

    ref_point = get_text(parcel_elem, "CP:referencePoint/gml:Point/gml:pos")
    if ref_point:
        coords = ref_point.split()
        if len(coords) >= 2:
            parcel_data["refPoint_x"] = float(coords[0]); parcel_data["refPoint_y"] = float(coords[1])
            lon, lat = convert_to_gps(float(coords[0]), float(coords[1]))
            parcel_data["refPoint_lon"], parcel_data["refPoint_lat"] = lon, lat

# === OPRAVENO: Získání druhu pozemku (Web Scraping z Nahlížení do KN) ===
    # ČÚZK ve svých bezplatných ArcGIS vrstvách druh pozemku neposkytuje.
    # Skript proto "nasimuluje" prohlížeč a přečte to přímo z webu KN.
    druh_pozemku_text = "Nezjištěno"
    try:
        # Vytáhneme čisté ID parcely z dříve stažených INSPIRE dat
        raw_id = parcel_data.get("inspire_localId", "") or parcel_data.get("gml_id", "")
        match_id = re.search(r'\d+', raw_id)
        
        if match_id:
            parcel_kod = match_id.group()
            url_kn = f"https://nahlizenidokn.cuzk.gov.cz/ZobrazObjekt.aspx?&typ=parcela&id={parcel_kod}"
            
            # Nasimulujeme běžný prohlížeč, aby nás server ČÚZK nezařízl
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            }
            res_kn = session.get(url_kn, headers=headers, timeout=10)
            
            if res_kn.status_code == 200:
                # Regulární výraz - hledá text za nadpisem "Druh pozemku:" v následující buňce tabulky
                match_druh = re.search(r'Druh pozemku:[\s\S]*?<td[^>]*>(.*?)</td>', res_kn.text, re.IGNORECASE)
                if match_druh:
                    # Očištění o případné skryté HTML tagy a prázdné znaky
                    cisty_text = re.sub(r'<[^>]+>', '', match_druh.group(1)).strip()
                    druh_pozemku_text = cisty_text
                    
    except Exception as e:
        print(f"Nepodařilo se stáhnout druh pozemku z webu KN pro {parcel_number}: {e}")

    parcel_data["druh_pozemku"] = druh_pozemku_text
    # ====================================================================

    return pd.DataFrame([parcel_data])

# =============================================================================
# 3. HLAVNÍ BLOK: ZPRACOVÁNÍ VSTUPŮ A VÝSTUP
# =============================================================================

parcely = [
    ("Praha", "Hostivař", "1714/4", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/5", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/6", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),    
    ("Praha", "Hostivař", "1714/7", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    ("Praha", "Hostivař", "1714/8", "1594", "Historicky oceňované další pozemky ZP 1300-15-2009, Ing.Arch. Soukeník"),
    
    ("Praha", "Hostivař", "1714/3", "804","Oceňovaný pozemek"),

    ("Praha", "Strašnice", "4241/9", "x1","Test pozemek s cenou"),
    ("Praha", "Strašnice", "4174/1", "x3","Test pozemek s cenou"),        
    ("Praha", "Hostivař", "2648/1", "x4","Test pozemek s cenou"),        
    ("Praha", "Strašnice", "4302/235", "x5","Test pozemek s cenou"),        
]

dfs = []
print("Načítám data z API ČÚZK...")

for okres, ku, parc, lv, info in parcely:
    df_one = get_parcel_data(okres, ku, parc)
    df_one["LV"] = str(lv) 
    df_one["info"] = str(info)
    dfs.append(df_one)

df_parcel_data = pd.concat(dfs, ignore_index=True)

df_export = (
    df_parcel_data
    .assign(
        parcelni_cislo=lambda d: d["label"],
        lat=lambda d: d["refPoint_lat"],
        lon=lambda d: d["refPoint_lon"],
    )
    .loc[:, ["okres_nazev", "ku_nazev", "obec_nazev", "parcelni_cislo", "LV", "info", "lat", "lon"]]
    .rename(columns={
        "okres_nazev": "okres",
        "ku_nazev": "katastralni_uzemi",
        "obec_nazev": "obec",
    })
)

df_export["lat"] = df_export["lat"].round(8)
df_export["lon"] = df_export["lon"].round(8)

out_path = "parcely_gps.xlsx"
df_export.to_excel(out_path, index=False, sheet_name="parcely_gps")
print(f"Data uložena do: {out_path}")

print("Generuji mapu...")
m = plot_parcels_on_map(df_parcel_data)

html_file = "mapa_parcel.html"
m.save(html_file)
print(f"Interaktivní mapa uložena do: {html_file}")

# Vygeneruje čisté HTML z objektu mapy a vynutí jeho zobrazení pod buňkou ve VS Code
display(HTML(m._repr_html_()))

Načítám data z API ČÚZK...
Data uložena do: parcely_gps.xlsx
Generuji mapu...
Interaktivní mapa uložena do: mapa_parcel.html
